In [73]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset, DataLoader

# Pre-Processing Data

## Functions

In [74]:
def load(fnm):
	return json.load(open(fnm))

def pre_proccess_data_from_choice_vs_no_choice(data):
    output = []

    for block_num, block in enumerate(data['blocks']):
        if block_num == 0:
            continue

        try:
            block_drift = block['block_config']['params']['startCameraMode']
        except (KeyError, TypeError):
            block_drift = 0
        
        trials = block['trials']

        for i in range(len(trials) // 3):
            try:
                choice_trial_sequence = [
                    trials[3*i]['hole_locations'],
                    trials[3*i+1]['hole_locations'],
                    trials[3*i+2]['hole_locations']
                ]
                
                chosen_path = [
                    trials[3*i]['events'][0]['holeUsed'], 
                    trials[3*i+1]['events'][0]['holeUsed'],
                    trials[3*i+2]['events'][0]['holeUsed']
                ]
                
                observed_rt = trials[3*i+2]['events'][0]['time'] - trials[3*i]['events'][0]['time']
                
                # does the middle trial have two options
                is_choice = len(trials[3*i+1]['hole_locations']) == 2

                if is_choice:
                    options = trials[3*i+1]['hole_locations']
                    chosen_hole = trials[3*i+1]['events'][0]['holeUsed']
                    
                    unchosen_hole = options[0] if options[0] != chosen_hole else options[1]
                    
                    non_chosen_path = [
                        trials[3*i]['events'][0]['holeUsed'], 
                        unchosen_hole, 
                        trials[3*i+2]['events'][0]['holeUsed']
                    ]
                    
                    chosen_1step_dist = abs(chosen_path[1] - chosen_path[0])
                    unchosen_1step_dist = abs(non_chosen_path[1] - non_chosen_path[0])
                    
                    chosen_2step_dist = chosen_1step_dist + abs(chosen_path[2] - chosen_path[1])
                    unchosen_2step_dist = unchosen_1step_dist + abs(non_chosen_path[2] - non_chosen_path[1])
                    
                else:
                    non_chosen_path = None
                    chosen_1step_dist = None
                    unchosen_1step_dist = None
                    chosen_2step_dist = None
                    unchosen_2step_dist = None
                
                output.append({
                    'block_number': block_num,
                    'trial_sequence_number': i,
                    'hole_sequence': choice_trial_sequence,
                    'chosen_path': chosen_path,
                    'non_chosen_path': non_chosen_path,
                    'observed_rt': observed_rt,
                    'choice_trial': is_choice,
                    'chosen_1step_dist': chosen_1step_dist,
                    'unchosen_1step_dist': unchosen_1step_dist,
                    'chosen_2step_dist': chosen_2step_dist,
                    'unchosen_2step_dist': unchosen_2step_dist,
                    'block_drift': block_drift
                })
            except (KeyError, IndexError, TypeError) as e:
                # If a row has broken events data, skip this individual sequence
                print(f"⚠️ Skipping step sequence {i} in block {block_num} due to processing error: {e}")
                continue

    output = pd.DataFrame(output)


    Q1 = output['observed_rt'].quantile(0.25)
    Q3 = output['observed_rt'].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 2.5 * IQR
    upper_bound = Q3 + 2.5 * IQR
    
    output = output[(output['observed_rt'] >= lower_bound) & (output['observed_rt'] <= upper_bound)]
    
    output = output.reset_index(drop=True)

    chosen_middle = output['chosen_path'].str[1]
    unchosen_middle = output['non_chosen_path'].str[1]

    output['chosen_left'] = (chosen_middle < unchosen_middle)
    return output
    return output

In [75]:
def prepare_rnn_tensors(raw_data, batch_size=1, test_split=0.2):
    """
    Processes raw maze data and splits it into chronologically separated 
    training and testing dataloaders based on a target percentage of total trials.
    """
    processed_data = pre_proccess_data_from_choice_vs_no_choice(raw_data)
    
    if isinstance(processed_data, list):
        df_raw = pd.DataFrame(processed_data)
    else:
        df_raw = processed_data
        
    is_left = df_raw['chosen_left'].astype(bool)
    
    L1 = np.where(is_left, df_raw['chosen_1step_dist'], df_raw['unchosen_1step_dist'])
    R1 = np.where(~is_left, df_raw['chosen_1step_dist'], df_raw['unchosen_1step_dist'])
    
    chosen_2step_diff = df_raw['chosen_2step_dist'] - df_raw['chosen_1step_dist']
    unchosen_2step_diff = df_raw['unchosen_2step_dist'] - df_raw['unchosen_1step_dist']
    
    L2 = np.where(is_left, chosen_2step_diff, unchosen_2step_diff)
    R2 = np.where(~is_left, chosen_2step_diff, unchosen_2step_diff)

    X = pd.DataFrame({
        'L1-R1': L1 - R1,
        'L2-R2': L2 - R2,
        'block_drift': df_raw['block_drift'],
        'block_number': df_raw['block_number'],
        'chosen_left': df_raw['chosen_left'],
        'cost': df_raw['observed_rt']
    })

    # Filter out practice rounds dynamically
    trials_per_block = X.groupby('block_number').size()
    large_blocks = trials_per_block[trials_per_block > 4].index
    
    if not large_blocks.empty:
        first_real_block = large_blocks.min()
        X = X[X['block_number'] >= first_real_block].copy()

    # Clean missing data
    for col in ['L1-R1', 'L2-R2', 'cost']:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        
    X = X.dropna(subset=['L1-R1', 'L2-R2', 'cost', 'chosen_left'])

    # Scale the time cost
    max_time = X['cost'].max()
    min_time = X['cost'].min()
    X['cost'] = (X['cost'] - min_time) / (max_time - min_time + 1e-6)

    # --- TRAIN/TEST SPLIT LOGIC ---
    # Recalculate valid trials per block after cleaning, sorted chronologically
    valid_trials_per_block = X.groupby('block_number').size().sort_index()
    
    if test_split > 0.0:
        cumulative_trials = valid_trials_per_block.cumsum()
        total_trials = cumulative_trials.iloc[-1]
        
        # Find the cutoff threshold (e.g., if split is 0.2, train threshold is 80% of trials)
        train_threshold = total_trials * (1 - test_split)
        
        train_blocks = set(valid_trials_per_block[cumulative_trials <= train_threshold].index)
        test_blocks = set(valid_trials_per_block[cumulative_trials > train_threshold].index)
        
        # Safety net: If test_split is very small but we still want at least 1 test block
        if len(test_blocks) == 0 and len(valid_trials_per_block) > 1:
            test_blocks = {valid_trials_per_block.index[-1]}
            train_blocks = set(valid_trials_per_block.index[:-1])
    else:
        train_blocks = set(valid_trials_per_block.index)
        test_blocks = set()

    feature_cols = ['L1-R1', 'L2-R2', 'block_drift', 'cost']

    # Helper function to generate identical tensor structures for both splits
    def build_tensors(target_blocks, is_train=True):
        features_list = []
        targets_list = []
        
        for b in sorted(target_blocks):
            group = X[X['block_number'] == b].sort_index()
            features_list.append(torch.tensor(group[feature_cols].values, dtype=torch.float32))
            targets_list.append(torch.tensor(group['chosen_left'].values, dtype=torch.long))
            
        if not features_list:
            return None, None, None
            
        x_pad = pad_sequence(features_list, batch_first=True, padding_value=0.0)
        y_pad = pad_sequence(targets_list, batch_first=True, padding_value=-1)
        
        dataset = TensorDataset(x_pad, y_pad)
        # We usually shuffle training blocks, but keep test blocks sequential for easier evaluation tracking
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=is_train)
        return x_pad, y_pad, loader

    # Build the distinct loaders
    X_train, y_train, train_loader = build_tensors(train_blocks, is_train=True)
    X_test, y_test, test_loader = build_tensors(test_blocks, is_train=False)

    print("\n✅ Tensor Preparation Complete!")
    print(f"-> Extracted {len(train_blocks)} Training Blocks, {len(test_blocks)} Testing Blocks.")
    
    if X_train is not None:
        print(f"-> Train X_padded shape: {X_train.shape} | Train y_padded shape: {y_train.shape}")
    if X_test is not None:
        print(f"-> Test X_padded shape:  {X_test.shape} | Test y_padded shape:  {y_test.shape}")

    # Returns two tuples: one for train, one for test
    return (X_train, y_train, train_loader), (X_test, y_test, test_loader)

## Loading

In [76]:
data = load("cloud study data/65D6694BE06947289BE4336BC1DE271A-019e9464-b9d3-798d-aa65-c87d82961db6-019e8386-74e7-7359-827b-6b4e4bc47db9-2026-06-04T21-03-48-346Z-fg8d.json")

In [77]:
processed_data = pd.DataFrame(pre_proccess_data_from_choice_vs_no_choice(data))
processed_data = processed_data[processed_data['block_number'] > 4]

In [78]:
processed_data

,block_number,trial_sequence_number,hole_sequence,chosen_path,non_chosen_path,observed_rt,choice_trial,chosen_1step_dist,unchosen_1step_dist,chosen_2step_dist,unchosen_2step_dist,block_drift,chosen_left
4,5,0,"[[7], [2, 9], [5]]","[7, 9, 5]","[7, 2, 5]",1333.5,True,2,5,6,8,0,False
5,5,1,"[[5], [0, 10], [2]]","[5, 0, 2]","[5, 10, 2]",2801.1,True,5,5,7,13,0,True
6,5,2,"[[4], [2, 7], [4]]","[4, 7, 4]","[4, 2, 4]",2967.4,True,3,2,6,4,0,False
7,5,3,"[[5], [2, 11], [3]]","[5, 2, 3]","[5, 11, 3]",1950.7,True,3,6,4,14,0,True
8,5,4,"[[4], [2, 11], [6]]","[4, 2, 6]","[4, 11, 6]",2016.5,True,2,7,6,12,0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1063,36,28,"[[5], [2, 10], [4]]","[5, 2, 4]","[5, 10, 4]",898.8,True,3,5,5,11,1,True
1064,36,29,"[[8], [3, 11], [6]]","[8, 3, 6]","[8, 11, 6]",1300.0,True,5,3,8,8,1,True
1065,36,30,"[[9], [6, 11], [8]]","[9, 6, 8]","[9, 11, 8]",1916.6,True,3,2,5,5,1,True
1066,36,31,"[[8], [6, 11], [8]]","[8, 6, 8]","[8, 11, 8]",867.8,True,2,3,4,6,1,True


In [79]:
X = pd.DataFrame({
    'L1': np.where(processed_data['chosen_left'], processed_data['chosen_1step_dist'], processed_data['unchosen_1step_dist']),
    'R1': np.where(~processed_data['chosen_left'], processed_data['chosen_1step_dist'], processed_data['unchosen_1step_dist']),
    
    'L2': np.where(processed_data['chosen_left'], 
                   processed_data['chosen_2step_dist'] - processed_data['chosen_1step_dist'], 
                   processed_data['unchosen_2step_dist'] - processed_data['unchosen_1step_dist']),
    'R2': np.where(~processed_data['chosen_left'], 
                   processed_data['chosen_2step_dist'] - processed_data['chosen_1step_dist'], 
                   processed_data['unchosen_2step_dist'] - processed_data['unchosen_1step_dist']),

    'block_drift': processed_data['block_drift']
})

X = pd.DataFrame({
    'L1-R1': X['L1']-X['R1'],
    'L2-R2': X['L2']-X['R2'],
    'block_drift': X['block_drift'],
    'block_number': processed_data['block_number'],
    'chosen_left': processed_data['chosen_left'],
    'cost': processed_data['observed_rt']
})

In [80]:
X

,L1-R1,L2-R2,block_drift,block_number,chosen_left,cost
4,3,-1,0,5,False,1333.5
5,0,-6,0,5,True,2801.1
6,-1,-1,0,5,False,2967.4
7,-3,-7,0,5,True,1950.7
8,-5,-1,0,5,True,2016.5
...,...,...,...,...,...,...
1063,-2,-4,1,36,True,898.8
1064,2,-2,1,36,True,1300.0
1065,1,-1,1,36,True,1916.6
1066,-1,-1,1,36,True,867.8


# RNNs and Training

In [81]:
participant1 = load("cloud study data/65D6694BE06947289BE4336BC1DE271A-019e9464-b9d3-798d-aa65-c87d82961db6-019e8386-74e7-7359-827b-6b4e4bc47db9-2026-06-04T21-03-48-346Z-fg8d.json")
train_data, test_data = prepare_rnn_tensors(participant1, batch_size = 4, test_split= 0.2)
X_train, y_train, train_loader = train_data
X_test, y_test, test_loader = test_data


✅ Tensor Preparation Complete!
-> Extracted 25 Training Blocks, 7 Testing Blocks.
-> Train X_padded shape: torch.Size([25, 34, 4]) | Train y_padded shape: torch.Size([25, 34])
-> Test X_padded shape:  torch.Size([7, 34, 4]) | Test y_padded shape:  torch.Size([7, 34])


In [82]:
class TinyDecisionRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_actions):

        super(TinyDecisionRNN, self).__init__()
        
        self.hidden_size = hidden_size
        
        self.gru = nn.GRU(input_size=input_size, 
                          hidden_size=hidden_size, 
                          batch_first=True)

        self.readout = nn.Linear(in_features=hidden_size, 
                                 out_features=num_actions)

    def forward(self, x, h_0=None):

        gru_out, h_n = self.gru(x, h_0)
        
        logits = self.readout(gru_out)
        
        probabilities = torch.softmax(logits, dim=-1)
        
        return probabilities, h_n
    
def evaluate_model_performance(model, data_loader):
    model.eval()  # Switch model to evaluation mode
    
    total_log_likelihood = 0.0
    all_predictions = []
    all_actuals = []
    
    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            h_0 = torch.zeros(1, batch_x.size(0), model.hidden_size)
            probabilities, _ = model(batch_x, h_0)
            
            for run_idx in range(batch_x.size(0)):
                run_probs = probabilities[run_idx].view(-1, model.readout.out_features)
                run_actuals = batch_y[run_idx].view(-1)
                
                for step_idx in range(run_probs.size(0)):
                    actual_action = run_actuals[step_idx].item()
                    
                    if actual_action == -1:
                        break
                        
                    if actual_action not in [0, 1]:
                        continue
                    
                    try:
                        chosen_prob = run_probs[step_idx][actual_action].item()

                        if np.isnan(chosen_prob) or chosen_prob == 0.0:
                            print(f"Something\'s wrong @ {run_idx}, Step {step_idx}!")
                        total_log_likelihood += np.log(max(chosen_prob, 1e-7))
                        
                        predicted_action = torch.argmax(run_probs[step_idx]).item()
                        
                        all_predictions.append(predicted_action)
                        all_actuals.append(actual_action)
                        
                    except IndexError as e:
                        print(f"Index alignment error at step {step_idx}: {e}")
                        continue
                    
    all_actuals = np.array(all_actuals)
    all_predictions = np.array(all_predictions)
           
    total_steps = len(all_actuals)
    accuracy = np.sum(all_predictions == all_actuals) / total_steps
    err_matrix = confusion_matrix(all_actuals, all_predictions, labels=[0, 1])

    
    return {
        "log_likelihood": total_log_likelihood,
        "accuracy": accuracy,
        "error_matrix": err_matrix
    }

In [83]:
#X_train, y_train, train_loader = prepare_rnn_tensors(participant1, batch_size = 4)

In [84]:
learning_rate = 0.005 
l1_lambda = 1e-4
num_epochs = 150

model = TinyDecisionRNN(input_size=4, hidden_size=2, num_actions=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# Use CrossEntropyLoss and specify ignore_index=-1 so the loss calculation ignores the trailing padding zeros we added to unequal sequences
criterion = torch.nn.CrossEntropyLoss(ignore_index=-1)

for epoch in range(20):
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()

        h_0 = torch.zeros(1, batch_x.size(0), 2)
        
        gru_out, _ = model.gru(batch_x, h_0)
        
        logits = model.readout(gru_out) 
        
        logits = logits.view(-1, 2)
        batch_y = batch_y.view(-1)
        
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()



In [85]:
feature_names = ['L1-R1', 'L2-R2', 'block_drift']
evaluate_model_performance(model, test_loader)

{'log_likelihood': np.float64(-104.26983755088172),
 'accuracy': np.float64(0.7974137931034483),
 'error_matrix': array([[87, 31],
        [16, 98]])}

In [86]:
participant2 = load("cloud study data/88AD64F00C6B43489770A02E7A1AE2C2-019e8fd9-16e9-7876-8e3b-d51a48df0526-019e8386-74e7-7359-827b-6b4e4bc47db9-2026-06-03T23-37-31-300Z-4ecm.json")
train_data, test_data = prepare_rnn_tensors(participant2, batch_size=4, test_split=0.2)
X_train, y_train, train_loader = train_data
X_test, y_test, test_loader = test_data


⚠️ Skipping step sequence 31 in block 9 due to processing error: list index out of range
⚠️ Skipping step sequence 32 in block 9 due to processing error: list index out of range
⚠️ Skipping step sequence 29 in block 14 due to processing error: list index out of range
⚠️ Skipping step sequence 30 in block 14 due to processing error: list index out of range
⚠️ Skipping step sequence 31 in block 14 due to processing error: list index out of range
⚠️ Skipping step sequence 32 in block 14 due to processing error: list index out of range
⚠️ Skipping step sequence 19 in block 21 due to processing error: list index out of range
⚠️ Skipping step sequence 20 in block 21 due to processing error: list index out of range
⚠️ Skipping step sequence 21 in block 21 due to processing error: list index out of range
⚠️ Skipping step sequence 25 in block 34 due to processing error: list index out of range
⚠️ Skipping step sequence 26 in block 34 due to processing error: list index out of range
⚠️ Skipping 

In [87]:
learning_rate = 0.005 
l1_lambda = 1e-4
num_epochs = 150

model2 = TinyDecisionRNN(input_size=4, hidden_size=2, num_actions=2)
optimizer2 = torch.optim.Adam(model.parameters(), lr=0.005)

# Use CrossEntropyLoss and specify ignore_index=-1 so the loss calculation ignores the trailing padding zeros we added to unequal sequences
criterion = torch.nn.CrossEntropyLoss(ignore_index=-1)

for epoch in range(20):
    for batch_x, batch_y in train_loader:
        optimizer2.zero_grad()

        h_0 = torch.zeros(1, batch_x.size(0), 2)
        
        gru_out, _ = model2.gru(batch_x, h_0)
        
        logits = model2.readout(gru_out) 
        
        logits = logits.view(-1, 2)
        batch_y = batch_y.view(-1)
        
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer2.step()

In [88]:
evaluate_model_performance(model2, test_loader)

{'log_likelihood': np.float64(-185.1743905612916),
 'accuracy': np.float64(0.41201716738197425),
 'error_matrix': array([[ 65,  34],
        [103,  31]])}